## RAG PIPELINE - DATA INGESTION TO VECTOR DB PIPELINE

In [2]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/var/folders/bq/ydkh51gs3n985p18fxm390840000gn/T/ipykernel_3807/3887852148.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
/Users/nakulbachani/Documents/Applied_ML_Task1/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
## Read all the pdfs inside the directory
def process_all_pdf(directory):
    """Process all PDF files in the specified directory and return a list of documents."""
    all_documents = []
    pdf_dir = Path(directory)

    pdf_files = list(pdf_dir.glob("*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in the directory:")

    for pdf_file in pdf_files:
        print(f"Processing: {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            for info in documents:
                info.metadata["sorce_file"] = pdf_file.name
                info.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} documents from {pdf_file.name}")

        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# process all pdfs in the directory
all_pdf_documents = process_all_pdf("../data/pdf")
       

Found 7 PDF files in the directory:
Processing: Chat_Models.pdf
Loaded 77 documents from Chat_Models.pdf
Processing: LoRA.pdf
Loaded 26 documents from LoRA.pdf
Processing: RAG.pdf
Loaded 19 documents from RAG.pdf
Processing: LLM_Models.pdf
Loaded 75 documents from LLM_Models.pdf
Processing: BERT.pdf
Loaded 16 documents from BERT.pdf
Processing: Sentene_BERT.pdf
Loaded 11 documents from Sentene_BERT.pdf
Processing: Attention_is_all_you_need.pdf
Loaded 15 documents from Attention_is_all_you_need.pdf

Total documents loaded: 239


In [5]:
all_pdf_documents


[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'source': '../data/pdf/Chat_Models.pdf', 'file_path': '../data/pdf/Chat_Models.pdf', 'total_pages': 77, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'trapped': '', 'modDate': 'D:20230720003036Z', 'creationDate': 'D:20230720003036Z', 'page': 0, 'sorce_file': 'Chat_Models.pdf', 'file_type': 'pdf'}, page_content='Llama 2: Open Foundation and Fine-Tuned Chat Models\nHugo Touvron∗\nLouis Martin†\nKevin Stone†\nPeter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra\nPrajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen\nGuillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller\nCynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou\nHakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel

In [6]:
## Text splitting get into chunks

def split_docs(documents, chunk_size=1000,chunk_overlap =200):
    """Split documents into smaller chunks using RecursiveCharacterTextSplitter."""
    txt_split = RecursiveCharacterTextSplitter(chunk_size=chunk_size,
                                                chunk_overlap=chunk_overlap,
                                                length_function=len,
                                                separators=["\n\n", "\n", " ", ""])
    split_docs = txt_split.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    #show example of a chunk
    if split_docs:
        print("\nExample chunk:")
        print(f"Content : {split_docs[0].page_content[:500]}")
        print(f"Metadata : {split_docs[0].metadata}")  # Print the metadata of the first chunk
    
    return split_docs

In [7]:
chunks = split_docs(all_pdf_documents)
chunks

Split 239 documents into 1056 chunks.

Example chunk:
Content : Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗
Louis Martin†
Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Metadata : {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'source': '../data/pdf/Chat_Models.pdf', 'file_path': '../data/pdf/Chat_Models.pdf', 'total_pages': 77, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'trapped': '', 'modDate': 'D:20230720003036Z', 'creationDate': 'D:20230720003036Z', 'page':

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'source': '../data/pdf/Chat_Models.pdf', 'file_path': '../data/pdf/Chat_Models.pdf', 'total_pages': 77, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'trapped': '', 'modDate': 'D:20230720003036Z', 'creationDate': 'D:20230720003036Z', 'page': 0, 'sorce_file': 'Chat_Models.pdf', 'file_type': 'pdf'}, page_content='Llama 2: Open Foundation and Fine-Tuned Chat Models\nHugo Touvron∗\nLouis Martin†\nKevin Stone†\nPeter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra\nPrajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen\nGuillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller\nCynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou\nHakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel

# EMBEDDING AND VECTOR DB

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer."""

    def __init__(self,model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding model."""
        #model_name: Hugging Face model name for generating embeddings. Default is "all-MiniLM-L6-v2
        self.model_name = model_name
        self.model = None
        self._load_model()
        print(f"Initialized SentenceTransformer with model: {model_name}")

    def _load_model(self):
        """Load the SentenceTransformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print("Model loaded successfully. Embedding dimension:", self.model.get_embedding_dimension())
        except Exception as e:
            print(f"Error occurred while loading the model: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray: #texts gives string , instead of chunk object
        """Generate embeddings for a list of texts."""
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        
        try:
            print(f"Generating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"Embeddings generated successfully. with shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Error occurred during embedding generation: {e}")
            raise

#initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9083.73it/s]


Model loaded successfully. Embedding dimension: 384
Initialized SentenceTransformer with model: all-MiniLM-L6-v2


## VECTOR STORE

In [15]:
class VectorStore:
    """manages document embeddings in a chromadb vector database."""

    def __init__(self, collection_name: str = "pdf_documents",persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store.

        Args:
            collection_name: Name of the ChromaDB collection to store document embeddings.
            persist_directory: Directory where the ChromaDB database will be persisted.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()


    def _initialize_store(self):
        """Initialize the ChromaDB client and collection."""
        try:
            print(f"Initializing ChromaDB client with persist directory: {self.persist_directory}")
            os.makedirs(self.persist_directory, exist_ok=True) # Ensure the persist directory exists
            print("Persist directory:")
            print(os.path.abspath(self.persist_directory))
            self.client = chromadb.PersistentClient(path=self.persist_directory)     # Create a persistent ChromaDB client that will store data in the specified directory)
            #Remote = Client ,TV = ChromaDB
            #Without a client: self.collection = ... , wouldn't work because Python doesn't know how to communicate with ChromaDB.
            
            #create or get collection
            self.collection = self.client.get_or_create_collection(name=self.collection_name,
                                                                   metadata={"description": "PDF document embedding for RAG"})
             #get_or_create_collection will return the collection if it exists, otherwise it will create a new collection with the specified name
            print(f"vector store initialized with collection: {self.collection_name}")
            print(f"existing document in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing ChromaDB client or collection: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray): # documents = chunks 
        """
        Add documents and their corresponding embeddings to the vector store.

        Args:
            documents: List of document objects (e.g., LangChain Document) to be stored.
            embeddings: Numpy array of shape (len(documents), embedding_dimension) containing the embeddings for the documents.
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        print(f"Adding {len(documents)} documents to the vector store...")

        #prepare data for chromadb
        ids = []
        metadatas = []
        document_text = []
        embeddings_list = []

        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            #generate unique id for each document and prepare metadata and text for chromadb
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}" # Generate a unique ID for each document using uuid4
            ids.append(doc_id)

            #prepapre metadata
            metadata = dict(doc.metadata) # Start with the existing metadata from the document
            metadata['doc_index'] = i # Add the document index to the metadata
            metadata['content_length'] = len(doc.page_content) # Add content length to metadata
            metadatas.append(metadata)

            #document content
            document_text.append(doc.page_content)

            #embedding
            embeddings_list.append(embedding.tolist()) # Convert numpy array to list for JSON serialization

        #add to chromadb collection
        try:
            self.collection.add(ids=ids, metadatas=metadatas, documents=document_text, embeddings=embeddings_list)
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection after addition: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to the vector store: {e}")
            raise

vector_store = VectorStore()
vector_store

Initializing ChromaDB client with persist directory: ../data/vector_store
Persist directory:
/Users/nakulbachani/Documents/Applied_ML_Task1/data/vector_store
vector store initialized with collection: pdf_documents
existing document in collection: 0


In [16]:
import os

print(os.path.exists("../data/vector_store/chroma.sqlite3"))

True


In [17]:
print(vector_store.collection.count())

0


In [18]:
#convert the text to embeddings
texts = [doc.page_content for doc in chunks]

#generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

#store the documents and embeddings in the vector store
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 1056 texts...


Batches: 100%|██████████| 33/33 [00:06<00:00,  5.50it/s]


Embeddings generated successfully. with shape: (1056, 384)
Adding 1056 documents to the vector store...
Successfully added 1056 documents to the vector store.
Total documents in collection after addition: 1056


In [19]:
print(vector_store.collection.count())

1056


## Retriever Pipeline from VecotrStore

In [20]:
class RAGRetriever:
    """Handles query-based  retrieval from the vector store"""

    def __init__(self,vector_store:VectorStore , embedding_manager:EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector store: Vector store containing document embedding
            embedding_manager: manager for generating query embedding
        """ 
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0)-> List[Dict[str,Any]]:
        """
        Retrieve relevant documents for a query

        Args:
        query = the search query 
        top_k: no. of top results to return
        score_threshold: minimum similarity score threshold

        Returns:
        List of dictionaries containg retreved documents and metadata
        """
        print(f"Retrieving document for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        #generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #search in vector store
        try:
            print("Collection count:", self.vector_store.collection.count())
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            print("=" * 50)
            print("QUERY RESULTS")
            print(results)
            print("=" * 50)

            print(results)
            print(results.keys())

            #process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents , metadatas , distances)):
                    #convert distances to similarity socre( chromadb uses cosine distances)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'distance': distance,
                            'similarity_score': similarity_score,
                            'rank': i+1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("no document found")

            return retrieved_docs
        except Exception as e:
            print(f"error during retrieval: {e}")
            return[]
        
rag_retriever = RAGRetriever(vector_store, embedding_manager)

In [21]:
rag_retriever

In [22]:
rag_retriever.retrieve("How does self-attention differ from recurrence?")

Retrieving document for query: 'How does self-attention differ from recurrence?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.68it/s]

Embeddings generated successfully. with shape: (1, 384)
Collection count: 1056
QUERY RESULTS
{'ids': [['doc_fd4c660f_1028', 'doc_1cb00138_1026', 'doc_e8ec8470_1021', 'doc_98311c22_1011', 'doc_9b7a8fb7_1027']], 'embeddings': None, 'documents': [['executed operations, whereas a recurrent layer requires O(n) sequential operations. In terms of\ncomputational complexity, self-attention layers are faster than recurrent layers when the sequence\n6', 'PEpos.\nWe also experimented with using learned positional embeddings [9] instead, and found that the two\nversions produced nearly identical results (see Table 3 row (E)). We chose the sinusoidal version\nbecause it may allow the model to extrapolate to sequence lengths longer than the ones encountered\nduring training.\n4\nWhy Self-Attention\nIn this section we compare various aspects of self-attention layers to the recurrent and convolu-\ntional layers commonly used for mapping one variable-length sequence of symbol representations\n(x1, ..., 

[{'id': 'doc_fd4c660f_1028',
  'content': 'executed operations, whereas a recurrent layer requires O(n) sequential operations. In terms of\ncomputational complexity, self-attention layers are faster than recurrent layers when the sequence\n6',
  'metadata': {'creator': 'LaTeX with hyperref',
   'total_pages': 15,
   'creationdate': '2024-04-10T21:11:43+00:00',
   'producer': 'pdfTeX-1.40.25',
   'trapped': '',
   'moddate': '2024-04-10T21:11:43+00:00',
   'modDate': 'D:20240410211143Z',
   'format': 'PDF 1.5',
   'page': 5,
   'keywords': '',
   'file_path': '../data/pdf/Attention_is_all_you_need.pdf',
   'content_length': 196,
   'source': '../data/pdf/Attention_is_all_you_need.pdf',
   'subject': '',
   'sorce_file': 'Attention_is_all_you_need.pdf',
   'title': '',
   'doc_index': 1028,
   'file_type': 'pdf',
   'creationDate': 'D:20240410211143Z',
   'author': ''},
  'distance': 0.747265100479126,
  'similarity_score': 0.252734899520874,
  'rank': 1},
 {'id': 'doc_1cb00138_1026',
  

In [24]:
print(len(chunks))

1056


In [25]:
embedding_manager.model

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

In [26]:
rag_retriever.retrieve("How does self-attention differ from recurrence?")

Retrieving document for query: 'How does self-attention differ from recurrence?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.55it/s]

Embeddings generated successfully. with shape: (1, 384)
Collection count: 1056
QUERY RESULTS
{'ids': [['doc_fd4c660f_1028', 'doc_1cb00138_1026', 'doc_e8ec8470_1021', 'doc_98311c22_1011', 'doc_9b7a8fb7_1027']], 'embeddings': None, 'documents': [['executed operations, whereas a recurrent layer requires O(n) sequential operations. In terms of\ncomputational complexity, self-attention layers are faster than recurrent layers when the sequence\n6', 'PEpos.\nWe also experimented with using learned positional embeddings [9] instead, and found that the two\nversions produced nearly identical results (see Table 3 row (E)). We chose the sinusoidal version\nbecause it may allow the model to extrapolate to sequence lengths longer than the ones encountered\nduring training.\n4\nWhy Self-Attention\nIn this section we compare various aspects of self-attention layers to the recurrent and convolu-\ntional layers commonly used for mapping one variable-length sequence of symbol representations\n(x1, ..., 

[{'id': 'doc_fd4c660f_1028',
  'content': 'executed operations, whereas a recurrent layer requires O(n) sequential operations. In terms of\ncomputational complexity, self-attention layers are faster than recurrent layers when the sequence\n6',
  'metadata': {'keywords': '',
   'page': 5,
   'producer': 'pdfTeX-1.40.25',
   'creationDate': 'D:20240410211143Z',
   'sorce_file': 'Attention_is_all_you_need.pdf',
   'creator': 'LaTeX with hyperref',
   'creationdate': '2024-04-10T21:11:43+00:00',
   'trapped': '',
   'title': '',
   'subject': '',
   'format': 'PDF 1.5',
   'file_path': '../data/pdf/Attention_is_all_you_need.pdf',
   'source': '../data/pdf/Attention_is_all_you_need.pdf',
   'total_pages': 15,
   'author': '',
   'modDate': 'D:20240410211143Z',
   'content_length': 196,
   'file_type': 'pdf',
   'doc_index': 1028,
   'moddate': '2024-04-10T21:11:43+00:00'},
  'distance': 0.747265100479126,
  'similarity_score': 0.252734899520874,
  'rank': 1},
 {'id': 'doc_1cb00138_1026',
  

In [27]:
rag_retriever.retrieve("What problem does RAG solve?")

Retrieving document for query: 'What problem does RAG solve?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.73it/s]

Embeddings generated successfully. with shape: (1, 384)
Collection count: 1056
QUERY RESULTS
{'ids': [['doc_cc6a8c3d_502', 'doc_52bbf45f_537', 'doc_b474df41_495', 'doc_0c94a21d_543', 'doc_ffe3ba78_484']], 'embeddings': None, 'documents': [['Broader Impact\nThis work offers several positive societal beneﬁts over previous work: the fact that it is more\nstrongly grounded in real factual knowledge (in this case Wikipedia) makes it “hallucinate” less\nwith generations that are more factual, and offers more control and interpretability. RAG could be\nemployed in a wide variety of scenarios with direct beneﬁt to society, for example by endowing it\nwith a medical index and asking it open-domain questions on that topic, or by helping people be more\neffective at their jobs.\nWith these advantages also come potential downsides: Wikipedia, or any potential external knowledge\nsource, will probably never be entirely factual and completely devoid of bias. Since RAG can be\nemployed as a language 

[]

## Integrating VectorDB context pipeline with LLM output

In [29]:
## Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

## initialize the groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024)

#Simple RAG function : retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=5):

    # Retrieve relevant chunks
    results = retriever.retrieve(query, top_k=top_k)

    context = "\n\n".join(
        [doc["content"] for doc in results]
    ) if results else ""

    if not context:
        return "No relevant context found to answer the question."

    # Prompt
    prompt = f"""
You are a helpful research assistant.

Answer ONLY using the provided context.

If the answer is not present in the context, say:
'I could not find the answer in the provided documents.'

Context:
{context}

Question:
{query}

Answer:
"""

    # Generate answer
    response = llm.invoke(prompt)
    answer = response.content

    # Collect source information
    sources = []

    for doc in results:

        metadata = doc["metadata"]

        filename = (
            metadata.get("source_file")
            or metadata.get("sorce_file")
            or "Unknown File"
        )

        page = metadata.get("page", "Unknown")

        similarity = doc.get("similarity_score", 0.0)

        sources.append({
            "file": filename,
            "page": page,
            "similarity": similarity
        })

    # Remove duplicates and keep highest similarity
    unique_sources = {}

    for source in sources:

        key = (source["file"], source["page"])

        if (
            key not in unique_sources
            or source["similarity"]
            > unique_sources[key]["similarity"]
        ):
            unique_sources[key] = source

    unique_sources = sorted(
        unique_sources.values(),
        key=lambda x: x["similarity"],
        reverse=True
    )

    # Final output
    final_response = f"""
Answer:
{answer}

Sources:
"""

    for i, source in enumerate(unique_sources, start=1):

        final_response += (
            f"\n{i}. {source['file']} (Page {source['page']})"
            f"\n   Similarity: {source['similarity']:.3f}\n"
        )

    return final_response

In [30]:
answer = rag_simple("How does self-attention differ from recurrence?",rag_retriever,llm)
print(answer)

Retrieving document for query: 'How does self-attention differ from recurrence?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.33it/s]

Embeddings generated successfully. with shape: (1, 384)
Collection count: 1056
QUERY RESULTS
{'ids': [['doc_fd4c660f_1028', 'doc_1cb00138_1026', 'doc_e8ec8470_1021', 'doc_98311c22_1011', 'doc_9b7a8fb7_1027']], 'embeddings': None, 'documents': [['executed operations, whereas a recurrent layer requires O(n) sequential operations. In terms of\ncomputational complexity, self-attention layers are faster than recurrent layers when the sequence\n6', 'PEpos.\nWe also experimented with using learned positional embeddings [9] instead, and found that the two\nversions produced nearly identical results (see Table 3 row (E)). We chose the sinusoidal version\nbecause it may allow the model to extrapolate to sequence lengths longer than the ones encountered\nduring training.\n4\nWhy Self-Attention\nIn this section we compare various aspects of self-attention layers to the recurrent and convolu-\ntional layers commonly used for mapping one variable-length sequence of symbol representations\n(x1, ..., 


Answer:
Self-attention differs from recurrence in that it connects all positions with a constant number of sequentially executed operations, whereas a recurrent layer requires O(n) sequential operations. This means that self-attention can be parallelized more easily and has a shorter path length between long-range dependencies, making it easier to learn such dependencies.

Sources:

1. Attention_is_all_you_need.pdf (Page 5)
   Similarity: 0.253

2. Attention_is_all_you_need.pdf (Page 4)
   Similarity: 0.093

3. Attention_is_all_you_need.pdf (Page 1)
   Similarity: 0.020



In [33]:
answer = rag_simple("Difference between BERT and GPT?",rag_retriever,llm)
print(answer)

Retrieving document for query: 'Difference between BERT and GPT?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.68it/s]

Embeddings generated successfully. with shape: (1, 384)
Collection count: 1056
QUERY RESULTS
{'ids': [['doc_ed90cea5_933', 'doc_e11c3143_927', 'doc_3dd69d69_876', 'doc_be8d30c8_622', 'doc_c79873ac_890']], 'embeddings': None, 'documents': [['approach.\nThe most comparable existing pre-training\nmethod to BERT is OpenAI GPT, which trains a\nleft-to-right Transformer LM on a large text cor-\npus. In fact, many of the design decisions in BERT\nwere intentionally made to make it as close to\nGPT as possible so that the two methods could be\nminimally compared. The core argument of this\nwork is that the bi-directionality and the two pre-\ntraining tasks presented in Section 3.1 account for\nthe majority of the empirical improvements, but\nwe do note that there are several other differences\nbetween how BERT and GPT were trained:\n• GPT is trained on the BooksCorpus (800M\nwords); BERT is trained on the BooksCor-\npus (800M words) and Wikipedia (2,500M\nwords).\n• GPT uses a sentence separat


Answer:
The differences between BERT and GPT are:

1. BERT uses a bidirectional Transformer, while GPT uses a left-to-right Transformer.
2. BERT is trained on both the BooksCorpus (800M words) and Wikipedia (2,500M words), while GPT is trained only on the BooksCorpus (800M words).
3. BERT learns [SEP], [CLS], and sentence A/B embeddings during pre-training, while GPT introduces these tokens only at fine-tuning time.
4. BERT was trained for a different number of steps and batch size compared to GPT (although the exact details for BERT's training are not provided in the given context).

Sources:

1. BERT.pdf (Page 13)
   Similarity: 0.333

2. BERT.pdf (Page 12)
   Similarity: 0.287

3. BERT.pdf (Page 2)
   Similarity: 0.144

4. LLM_Models.pdf (Page 17)
   Similarity: 0.092

5. BERT.pdf (Page 5)
   Similarity: 0.067



In [34]:
answer = rag_simple("how does LoRA reduce training cost?",rag_retriever,llm)
print(answer)

Retrieving document for query: 'how does LoRA reduce training cost?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 24.09it/s]

Embeddings generated successfully. with shape: (1, 384)
Collection count: 1056
QUERY RESULTS
{'ids': [['doc_a32f3616_361', 'doc_b10157d3_394', 'doc_5f871869_373', 'doc_b44d0bf6_437', 'doc_0ed0ac06_365']], 'embeddings': None, 'documents': [['and do not tune it. This scaling helps to reduce the need to retune hyperparameters when we vary\nr (Yang & Hu, 2021).\nA Generalization of Full Fine-tuning.\nA more general form of ﬁne-tuning allows the training of\na subset of the pre-trained parameters. LoRA takes a step further and does not require the accumu-\nlated gradient update to weight matrices to have full-rank during adaptation. This means that when\napplying LoRA to all weight matrices and training all biases2, we roughly recover the expressive-\nness of full ﬁne-tuning by setting the LoRA rank r to the rank of the pre-trained weight matrices. In\nother words, as we increase the number of trainable parameters 3, training LoRA roughly converges\nto training the original model, while ada


Answer:
I could not find the answer in the provided documents.

Sources:

1. LoRA.pdf (Page 3)
   Similarity: 0.297

2. LoRA.pdf (Page 9)
   Similarity: 0.185

3. LoRA.pdf (Page 5)
   Similarity: 0.124

4. LoRA.pdf (Page 20)
   Similarity: 0.114



In [35]:
answer = rag_simple("What is self-attention?",rag_retriever,llm)
print(answer)

Retrieving document for query: 'What is self-attention?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.84it/s]

Embeddings generated successfully. with shape: (1, 384)
Collection count: 1056
QUERY RESULTS
{'ids': [['doc_c0ee7284_1016', 'doc_e8ec8470_1021', 'doc_1cb00138_1026', 'doc_c852fbbf_1053', 'doc_98311c22_1011']], 'embeddings': None, 'documents': [['3.2\nAttention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3', 'Applications of Attention in our Model\nThe Transformer uses multi-head attention in three different ways:\n• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,\nand the memory keys and values come from the output of the encoder. This allows every\nposition in the decoder to attend over all positions in the input sequence. This mimics the\ntypical encoder-decoder attention mechanisms in sequence-to-sequence models such as\n[38, 2, 9].\n• The encoder contains self-attention layers. In a s


Answer:
I could not find the answer in the provided documents.

Sources:

1. Attention_is_all_you_need.pdf (Page 2)
   Similarity: 0.014



In [36]:
answer = rag_simple(
    "How does self-attention differ from recurrence?",
    rag_retriever,
    llm
)

print(answer)

Retrieving document for query: 'How does self-attention differ from recurrence?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.93it/s]

Embeddings generated successfully. with shape: (1, 384)
Collection count: 1056
QUERY RESULTS
{'ids': [['doc_fd4c660f_1028', 'doc_1cb00138_1026', 'doc_e8ec8470_1021', 'doc_98311c22_1011', 'doc_9b7a8fb7_1027']], 'embeddings': None, 'documents': [['executed operations, whereas a recurrent layer requires O(n) sequential operations. In terms of\ncomputational complexity, self-attention layers are faster than recurrent layers when the sequence\n6', 'PEpos.\nWe also experimented with using learned positional embeddings [9] instead, and found that the two\nversions produced nearly identical results (see Table 3 row (E)). We chose the sinusoidal version\nbecause it may allow the model to extrapolate to sequence lengths longer than the ones encountered\nduring training.\n4\nWhy Self-Attention\nIn this section we compare various aspects of self-attention layers to the recurrent and convolu-\ntional layers commonly used for mapping one variable-length sequence of symbol representations\n(x1, ..., 


Answer:
Self-attention differs from recurrence in that it connects all positions with a constant number of sequentially executed operations, whereas a recurrent layer requires O(n) sequential operations. This means that self-attention can be parallelized more easily and has a shorter path length between long-range dependencies, making it easier to learn such dependencies.

Sources:

1. Attention_is_all_you_need.pdf (Page 5)
   Similarity: 0.253

2. Attention_is_all_you_need.pdf (Page 4)
   Similarity: 0.093

3. Attention_is_all_you_need.pdf (Page 1)
   Similarity: 0.020

